In [ ]:
from core.data_constructor import DataConstructor
from datetime import datetime

In [ ]:
# Loads the dataset from snowflake - retrieves lat / long via geocoding (nominatim) - performs further geo mappings.
data_constructor = DataConstructor(TCBB_enable=False, address_enabled=True)
df = data_constructor.create_feature_set()

This should generate following columns:
```
Index(['hoofddossier_id', 'hoofddossier_flag', 'factuurbedrag',
       'betaling_klant', 'bedrag', 'booking_code', 'booking_dut',
       'dossier_status', 'dossier_status_code', 'solvabiliteit_dossier',
       'dossier_solvabiliteit', 'creatiedatum_hoofddossier', 'datumboeking',
       'creatiedatum_dossierlijn', 'gerechterlijk_minnelijk',
       'bedrag_weighted', 'bedrag_cumulatief', 'gerechtelijk', 'y',
       'dossier_status_label_reg', 'dossier_status_label_cat', 'verweerder',
       'verweerder_id', 'verweerder_b2b_b2c', 'verweerder_leeftijd',
       'verweerder_leeftijdscategorie', 'gevangenis_key',
       'verweerder_postcode', 'verweerder_straat', 'verweerder_straat_clean',
       'verweerder_nr', 'verweerder_adres', 'verweerder_gemeente',
       'verweerder_provincie', 'verweerder_gewest',
       'verweerder_straat_clean_nunique', 'verweerder_postcode_nunique',
       'gevangenis_key_nunique', 'n_people_at_address', 'estimated_leeftijd',
       'business', 'adres', 'latitude', 'longitude', 'X_Lambert72',
       'Y_Lambert72', 'geometry', 'CD_YEAR', 'CD_MUNTY_REFNIS', 'CD_SECTOR',
       'MS_NBR_NON_ZERO_INC', 'MS_TOT_NET_TAXABLE_INC',
       'MS_AVG_TOT_NET_TAXABLE_INC', 'MS_MEDIAN_NET_TAXABLE_INC',
       'MS_INT_QUART_DIFF', 'MS_INT_QUART_COEFF', 'MS_INT_QUART_ASSYM',
       'TX_SECTOR_DESCR_NL', 'TX_SECTOR_DESCR_FR'],
      dtype='object')
```

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import joblib
import numpy as np

# drop dossier status columns, they are the labels 
# NOTE: you will probably need to edit dossiers_query.sql because it now only queries for dossiers that have concluded
# For inference, you want the dossiers that are still open (i would suggest new + had a change)
columns = [x for x in df.columns if x not in ["dossier_status_label_reg", "dossier_status_label_reg"]]
test = df[columns]
random_forest_classifier = joblib.load('random_forest_classifier.pkl')
y_hat = classifier.predict_proba(test)
# pure prediction
# 1 = solvabel
# 0 = insolvabel

# now remap to our stoplight
y_step = np.zeros(y_hat.shape) # = y_hat.copy()
y_step[y_hat > 0.25] += 1
y_step[y_hat > 0.5] += 1
y_step[y_hat > 0.75] += 1

# 0.0 - 0.25 = level 0 => 0.62 kans insolvabel
# 0.25 - 0.5 = level 1 => 0.50 kans op insolvabel
# 0.5 - 0.75 = level 2 => 0.41 kans op insolvabel
# 0.75 - 1.0 = level 3 => 0.21 kans op insolvabel

